# Pass 8 — Publication-Grade Synthesis

This notebook consolidates Passes 3B–7 into a publication package.

**Passes included**
- Pass 3B: locked BUSI cohort and VGG16 baseline
- Pass 4: localization robustness
- Pass 5: Grad-CAM attribution
- Pass 6: zero-shot BrEaST external validation
- Pass 7: preprocessing ablation and bootstrap uncertainty

**Important scope:** the classifier is conditional on ground-truth lesion localization. This is not an end-to-end detection-and-diagnosis system.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
R = ROOT / "results"
OUT = R / "pass8_publication"
TABLES = OUT / "tables"
FIGURES = OUT / "figures"
TEXT = OUT / "manuscript_text"
for d in [TABLES, FIGURES, TEXT]:
    d.mkdir(parents=True, exist_ok=True)

P3 = R / "pass3b_vgg16"
P4 = R / "pass4_localization_robustness"
P5 = R / "pass5_96_image_only_bootstrap_correction"
P6 = R / "pass6b_breast_zero_shot_external_validation"
P7 = R / "pass7_preprocessing_ablation"

def f(base, name):
    p = base / name
    return p if p.exists() else None

paths = {
    "p3_split": f(P3/"split", "Pass_3B_locked_split.csv"),
    "p3_metrics": f(P3/"metrics", "pass3b_metrics.csv"),
    "p4_summary": f(P4/"tables", "pass4_localization_robustness_summary.csv"),
    "p4_repeat": f(P4/"tables", "pass4_localization_robustness_metrics_by_repeat.csv"),
    "p5_overall": f(P5, "pass5_overall_bootstrap_ci.csv"),
    "p5_class": f(P5, "pass5_class_bootstrap_ci.csv"),
    "p5_status": f(P5, "pass5_correct_incorrect_bootstrap_ci.csv"),
    "p5_class_diff": f(P5, "pass5_malignant_minus_benign_bootstrap_difference.csv"),
    "p5_status_diff": f(P5, "pass5_incorrect_minus_correct_bootstrap_difference.csv"),
    "p5_threshold": f(P5, "pass5_threshold_bootstrap_ci.csv"),
    "p6_metrics": f(P6, "pass6b_external_metrics.csv"),
    "p6_boot": f(P6, "pass6b_bootstrap_ci.csv"),
    "p6_cal": f(P6, "pass6b_calibration_summary.csv"),
    "p6_rel": f(P6, "pass6b_reliability_table.csv"),
    "p6_pred": f(P6, "pass6b_predictions.csv"),
    "p7_metrics": f(P7/"tables", "pass7_ablation_metrics.csv"),
    "p7_boot": f(P7/"tables", "pass7_bootstrap_ci.csv"),
    "p7_pairs": f(P7/"tables", "pass7_pairwise_bootstrap_differences.csv"),
}

inventory = pd.DataFrame([
    {"input":k, "status":"OK" if v else "MISSING", "path":str(v) if v else ""}
    for k,v in paths.items()
])
inventory.to_csv(TABLES/"pass8_input_inventory.csv", index=False)
display(inventory)

## 1. Load completed results

In [ ]:
def load(k):
    if paths[k] is None:
        raise FileNotFoundError(f"Missing required Pass 8 input: {k}")
    return pd.read_csv(paths[k])

p3_split = load("p3_split")
p3_metrics = load("p3_metrics")
p4_summary = load("p4_summary")
p4_repeat = load("p4_repeat")
p5_overall = load("p5_overall")
p5_class = load("p5_class")
p5_status = load("p5_status")
p5_class_diff = load("p5_class_diff")
p5_status_diff = load("p5_status_diff")
p5_threshold = load("p5_threshold")
p6_metrics = load("p6_metrics")
p6_boot = load("p6_boot")
p6_cal = load("p6_cal")
p6_rel = load("p6_rel")
p6_pred = load("p6_pred")
p7_metrics = load("p7_metrics")
p7_boot = load("p7_boot")
p7_pairs = load("p7_pairs")
print("All required inputs loaded.")

## 2. Integrity audit

In [ ]:
ext = p6_metrics.iloc[0]

checks = {
    "BUSI total = 647": len(p3_split) == 647,
    "BUSI train = 455": (p3_split.split == "train").sum() == 455,
    "BUSI validation = 96": (p3_split.split == "validation").sum() == 96,
    "BUSI test = 96": (p3_split.split == "test").sum() == 96,
    "BrEaST total = 252": int(ext.n) == 252,
    "BrEaST benign = 154": int(ext.n_benign) == 154,
    "BrEaST malignant = 98": int(ext.n_malignant) == 98,
    "Pass 7 has four conditions": set(p7_metrics.condition) == {"raw_roi","minmax_roi","clahe_roi","minmax_clahe_roi"},
    "Pass 7 test n = 96": bool((p7_metrics.n == 96).all()),
    "Pass 7 bootstrap = 10000": bool((p7_boot.n_bootstrap_finite == 10000).all()),
    "Pass 7 paired bootstrap = 10000": bool((p7_pairs.n_bootstrap_finite == 10000).all()),
}
tn = ((p6_pred.true_label==0)&(p6_pred.predicted_label==0)).sum()
fp = ((p6_pred.true_label==0)&(p6_pred.predicted_label==1)).sum()
fn = ((p6_pred.true_label==1)&(p6_pred.predicted_label==0)).sum()
tp = ((p6_pred.true_label==1)&(p6_pred.predicted_label==1)).sum()
checks["External confusion matrix matches"] = [tn,fp,fn,tp] == [int(ext.tn),int(ext.fp),int(ext.fn),int(ext.tp)]

audit = pd.DataFrame([{"check":k,"passed":v} for k,v in checks.items()])
audit.to_csv(TABLES/"pass8_integrity_checks.csv", index=False)
display(audit)
assert audit.passed.all()

## 3. Publication tables

Pass 7 is the controlled four-condition ablation experiment. Pass 3B remains the original locked baseline audit and is not silently substituted for Pass 7.

In [ ]:
p7_metrics.to_csv(TABLES/"table1_internal_preprocessing_ablation.csv", index=False)
p7_boot.to_csv(TABLES/"table2_internal_bootstrap_ci.csv", index=False)
p7_pairs.to_csv(TABLES/"table3_internal_paired_bootstrap_differences.csv", index=False)

p4_summary.to_csv(TABLES/"table4_localization_robustness.csv", index=False)
p4_repeat.to_csv(TABLES/"table4b_localization_repeat_metrics.csv", index=False)

p5_overall.to_csv(TABLES/"table5_gradcam_overall.csv", index=False)
p5_class.to_csv(TABLES/"table6_gradcam_by_class.csv", index=False)
p5_status.to_csv(TABLES/"table7_gradcam_by_prediction_status.csv", index=False)
p5_class_diff.to_csv(TABLES/"table8_gradcam_class_difference.csv", index=False)
p5_status_diff.to_csv(TABLES/"table9_gradcam_status_difference.csv", index=False)
p5_threshold.to_csv(TABLES/"table10_gradcam_threshold_sensitivity.csv", index=False)

p6_metrics.to_csv(TABLES/"table11_external_metrics.csv", index=False)
p6_boot.to_csv(TABLES/"table12_external_bootstrap_ci.csv", index=False)
p6_cal.to_csv(TABLES/"table13_external_calibration_summary.csv", index=False)
p6_rel.to_csv(TABLES/"table14_external_reliability.csv", index=False)

print("Tables generated:", len(list(TABLES.glob("table*.csv"))) )

## 4. Reconcile Pass 3B and Pass 7 raw-ROI models

In [ ]:
r3 = p3_metrics[p3_metrics.condition=="raw_roi"].iloc[0]
r7 = p7_metrics[p7_metrics.condition=="raw_roi"].iloc[0]

reconciliation = pd.DataFrame([
    {"source":"Pass 3B raw ROI","accuracy":r3.accuracy,"sensitivity":r3.sensitivity,
     "specificity":r3.specificity,"roc_auc":r3.roc_auc,"tn":r3.tn,"fp":r3.fp,"fn":r3.fn,"tp":r3.tp},
    {"source":"Pass 7 raw ROI","accuracy":r7.accuracy,"sensitivity":r7.sensitivity,
     "specificity":r7.specificity,"roc_auc":r7.roc_auc,"tn":r7.tn,"fp":r7.fp,"fn":r7.fn,"tp":r7.tp},
])
reconciliation.to_csv(TABLES/"pass3b_vs_pass7_raw_roi_reconciliation.csv", index=False)
display(reconciliation)

## 5. Publication figures

In [ ]:
# Figure 1: preprocessing ablation
metrics = ["accuracy","sensitivity","specificity","f1","roc_auc","pr_auc"]
labels = ["Accuracy","Sensitivity","Specificity","F1","ROC-AUC","PR-AUC"]
conds = ["raw_roi","minmax_roi","clahe_roi","minmax_clahe_roi"]
d = p7_metrics.set_index("condition")
x = np.arange(len(metrics))
fig, ax = plt.subplots(figsize=(10,5.5))
for i,c in enumerate(conds):
    ax.bar(x+(i-1.5)*0.20, [d.loc[c,m] for m in metrics], 0.20,
           label=c.replace("_roi","").replace("_"," ").title())
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0,1.05)
ax.set_ylabel("Metric"); ax.set_title("Internal BUSI Test Performance Across ROI Preprocessing")
ax.legend(); fig.tight_layout()
fig.savefig(FIGURES/"figure1_internal_preprocessing.png", dpi=300)
plt.close(fig)

# Figure 2: localization robustness
pc = p4_summary.copy()
pert = next(c for c in ["perturbation_percent","perturbation","nominal_perturbation"] if c in pc.columns)
fig, ax = plt.subplots(figsize=(9,5.5))
for m in ["accuracy","sensitivity","specificity","f1","roc_auc"]:
    if m in pc.columns:
        ax.plot(pc[pert], pc[m], marker="o", label=m.replace("_"," ").upper())
ax.set_xlabel("Nominal localization perturbation (%)"); ax.set_ylabel("Metric")
ax.set_ylim(0,1.05); ax.set_title("Robustness to Simulated Localization Perturbation")
ax.legend(); fig.tight_layout()
fig.savefig(FIGURES/"figure2_localization_robustness.png", dpi=300)
plt.close(fig)

# Figure 3: external reliability
fig, ax = plt.subplots(figsize=(6.5,6))
ax.plot([0,1],[0,1],"--",label="Perfect calibration")
ax.scatter(p6_rel.mean_predicted_probability, p6_rel.observed_malignant_rate,
           s=p6_rel.n*8)
ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed malignant rate")
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_title("BrEaST External Reliability Diagram")
ax.legend(); fig.tight_layout()
fig.savefig(FIGURES/"figure3_external_reliability.png", dpi=300)
plt.close(fig)

# Figure 4: internal/external discrimination
fig, ax = plt.subplots(figsize=(7,5))
x = np.arange(2)
ax.bar(x-0.17,[r7.roc_auc,ext.roc_auc],0.34,label="ROC-AUC")
ax.bar(x+0.17,[r7.pr_auc,ext.pr_auc],0.34,label="PR-AUC")
ax.set_xticks(x); ax.set_xticklabels(["BUSI internal","BrEaST external"])
ax.set_ylim(0,1.05); ax.set_ylabel("Area under curve")
ax.set_title("Internal vs External Discrimination")
ax.legend(); fig.tight_layout()
fig.savefig(FIGURES/"figure4_internal_external_discrimination.png", dpi=300)
plt.close(fig)

print("Figures generated:", len(list(FIGURES.glob("*.png"))))

## 6. Manuscript-ready Results

In [ ]:
auc = p6_boot[p6_boot.metric=="roc_auc"].iloc[0]
prauc = p6_boot[p6_boot.metric=="pr_auc"].iloc[0]
cal = p6_cal.iloc[0]

results = f"""# Results

### Internal evaluation and preprocessing ablation

On the locked BUSI test cohort (n=96), the controlled Pass 7 raw lesion-ROI model achieved accuracy {r7.accuracy:.3f}, balanced accuracy {r7.balanced_accuracy:.3f}, sensitivity {r7.sensitivity:.3f}, specificity {r7.specificity:.3f}, F1 {r7.f1:.3f}, ROC-AUC {r7.roc_auc:.3f}, and PR-AUC {r7.pr_auc:.3f}. Min-Max normalization produced identical threshold-level classification and discrimination metrics while modestly improving probabilistic scoring. CLAHE-based preprocessing changed the operating characteristics at threshold 0.50, but paired bootstrap intervals did not establish a clear classification advantage.

### Localization robustness

Simulated nominal ±5%, ±10%, and ±15% perturbations of lesion bounding-box position and size reduced realized localization overlap while classification performance remained high across the evaluated perturbations. This supports robustness to the specific simulated localization errors tested, not validation of a real-world detector or segmentation system.

### Grad-CAM attribution

Grad-CAM analysis of the 96-image test cohort showed modest spatial overlap between thresholded attribution maps and the original lesion annotations. Malignant cases showed greater lesion-focused attribution on several measures. These results indicate partial spatial correspondence rather than precise lesion localization or causal explanation.

### Zero-shot external validation

On the independent BrEaST-Lesions-USG cohort (n=252; 154 benign and 98 malignant), evaluated without retraining, fine-tuning, threshold optimization, or calibration, the model achieved accuracy {ext.accuracy:.3f}, balanced accuracy {ext.balanced_accuracy:.3f}, sensitivity {ext.sensitivity:.3f}, specificity {ext.specificity:.3f}, F1 {ext.f1:.3f}, ROC-AUC {ext.roc_auc:.3f} (95% CI {auc.ci95_low:.3f}–{auc.ci95_high:.3f}), and PR-AUC {ext.pr_auc:.3f} (95% CI {prauc.ci95_low:.3f}–{prauc.ci95_high:.3f}). The confusion matrix contained {int(ext.tn)} true negatives, {int(ext.fp)} false positives, {int(ext.fn)} false negatives, and {int(ext.tp)} true positives.

External performance was lower than internal BUSI performance, indicating reduced cross-dataset generalization. Native probability outputs showed calibration error, with Brier score {cal.brier_score:.3f} and log loss {cal.log_loss:.3f}.
"""
(TEXT/"results_section.md").write_text(results)
print(results)

## 7. Manuscript-ready Methods and Limitations

In [ ]:
methods = """# Methods summary

A VGG16-based binary classifier was evaluated for benign versus malignant breast-ultrasound lesion classification using lesion-localized inputs. The BUSI binary cohort contained 647 images after exclusion of normal cases, with a locked split of 455 training, 96 validation, and 96 test images.

Lesion-localized inputs were generated from ground-truth lesion masks using a 15×15 elliptical morphological dilation, masking, bounding-box cropping, and resizing to 224×224 pixels. The classifier used an ImageNet-pretrained VGG16 backbone with frozen convolutional weights, global average pooling, a 256-unit ReLU dense layer, dropout of 0.5, and a sigmoid output. Training used Adam at 1×10^-4, training-only augmentation, early stopping on validation ROC-AUC, and learning-rate reduction on validation-ROC-AUC plateau.

Localization robustness used nominal ±5%, ±10%, and ±15% perturbations to lesion bounding-box position and size. Grad-CAM was evaluated against the original, non-dilated lesion annotation. Preprocessing ablation compared raw ROI, per-image Min-Max normalization, CLAHE, and Min-Max plus CLAHE. Uncertainty was quantified with 10,000 image-level bootstrap replicates and paired bootstrap comparisons.

Zero-shot external validation used a locked BrEaST-Lesions-USG cohort of 252 binary-labeled cases (154 benign, 98 malignant) with tumor masks. No external retraining, fine-tuning, threshold optimization, or calibration was performed. External uncertainty used 10,000 case-level bootstrap replicates.
"""
(TEXT/"methods_summary.md").write_text(methods)

limitations = """# Limitations

1. The classifier is conditional on ground-truth lesion localization and is not an end-to-end detection-and-diagnosis system.
2. The internal test cohort contains 96 images.
3. External validation shows reduced performance relative to internal BUSI performance.
4. External calibration was assessed without fitting a calibration model.
5. Localization robustness covers only the simulated bounding-box perturbations evaluated.
6. Grad-CAM is post-hoc attribution, not evidence of causal reasoning or clinical validity.
7. Small BIRADS strata should be treated as descriptive.
8. Multi-center prospective validation would be required before clinical deployment claims.
"""
(TEXT/"limitations.md").write_text(limitations)
print("Methods and limitations saved.")

## 8. Final claim audit

In [ ]:
claim_audit = pd.DataFrame([
    ["Lesion localization is important for internal discrimination","SUPPORTED"],
    ["Preprocessing materially improves discrimination","NOT SUPPORTED"],
    ["CLAHE is definitively superior","NOT SUPPORTED"],
    ["Robustness to the evaluated simulated localization errors","SUPPORTED WITH SCOPE"],
    ["Grad-CAM precisely localizes lesions","NOT SUPPORTED"],
    ["Perfect external generalization","NOT SUPPORTED"],
    ["Meaningful external discrimination","SUPPORTED"],
    ["Perfect external calibration","NOT SUPPORTED"],
    ["End-to-end clinical AI validation","NOT SUPPORTED"],
], columns=["claim","status"])
claim_audit.to_csv(TABLES/"pass8_claim_audit.csv", index=False)
display(claim_audit)

manifest = {
    "study":"Breast ultrasound lesion classification",
    "pass":8,
    "seed":42,
    "internal_binary_cohort":647,
    "internal_split":{"train":455,"validation":96,"test":96},
    "external_cohort":{"dataset":"BrEaST-Lesions-USG","n":252,"benign":154,"malignant":98},
    "primary_threshold":0.5,
    "bootstrap_replicates":10000,
    "ground_truth_roi":True,
    "external_zero_shot":True,
}
(OUT/"pass8_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))